# 05 - RFM Analysis (Mô hình hóa Hành vi Khách hàng)

**Mục tiêu đề tài (Q2):** *"Hành vi mua hàng của từng khách hàng có thể được mô tả như thế nào?"*

**Nguyên tắc cốt lõi:**
1. **CUSTOMER LEVEL:** RFM được xây dựng ở cấp độ khách hàng. Tuyệt đối không chạy gom cụm trực tiếp trên từng dòng giao dịch riêng lẻ.
2. **Định nghĩa các chiều:**
   - **R (Recency):** Số ngày từ lần mua cuối cùng đến ngày tham chiếu (`ReferenceDate`). (R càng nhỏ = mới mua gần đây).
   - **F (Frequency):** Số lượng hóa đơn mua hàng thành công khác nhau (`nunique(InvoiceNo)`).
   - **M (Monetary):** Tổng giá trị chi tiêu thuần của khách hàng sau khi trừ đơn hủy (`SUM(Revenue)`).
3. **RFM ≠ K-Means:** RFM là bước biểu diễn đặc trưng (Feature Representation), K-Means là thuật toán phân cụm (Clustering).

## 0. Chuẩn bị thư viện & Nạp module

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.modeling.rfm import build_rfm_table, summarize_rfm_distributions

pd.options.display.float_format = "{:.2f}".format

df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "retail_cleaned.csv")
print(f"Nạp thành công {len(df):,} dòng giao dịch sạch từ retail_cleaned.csv")

## 1. Xây dựng Bảng RFM (Recency, Frequency, Monetary)

- Loại bỏ các dòng thiếu `CustomerID` (vì không thể gán hành vi cho ai).
- Đặt `ReferenceDate` = ngày cuối cùng của tập dữ liệu + 1 ngày (`2011-12-10`).
- Loại bỏ các khách hàng có `Monetary <= 0` (mua trước kỳ quan sát rồi hoàn trả hết).

In [ ]:
rfm_df = build_rfm_table(df)
print("\n5 dòng đầu của bảng RFM:")
rfm_df.head(10)

## 2. Thống kê Mô tả & Phân tích Độ lệch (Skewness) của RFM

**Câu hỏi:** Phân phối của 3 biến R, F, M có đối xứng không hay bị lệch mạnh?

In [ ]:
rfm_summary = summarize_rfm_distributions(rfm_df)
rfm_summary

**Nhận xét quan trọng làm tiền đề cho Phase 8 (K-Means):**
- **Recency:** Trung vị là 51 ngày (50% khách hàng mua hàng trong vòng 51 ngày gần nhất). Hệ số Skewness = 1.25 (hơi lệch phải).
- **Frequency:** Trung vị chỉ là 2 hóa đơn, trong khi giá trị cao nhất lên tới 209 hóa đơn. Hệ số Skewness = 12.05 (lệch phải rất mạnh).
- **Monetary:** Trung vị là £653.35, trong khi trung bình là £1,920.72 và max lên tới £279,489.02. Hệ số Skewness = 21.59 (lệch phải cực kỳ mạnh do sự hiện diện của các đại lý mua buôn sỉ).

> 💡 **Kết luận chuẩn bị cho Phase 8:** Vì F và M có độ lệch phải rất cao và thang đo chênh lệch lớn giữa các chiều (R tính bằng ngày từ 1-374, F từ 1-209, M từ vài chục đến hàng trăm nghìn bảng), trước khi đưa vào thuật toán K-Means, chúng ta bắt buộc phải thực hiện **Log-transform** để giảm độ lệch, sau đó áp dụng **StandardScaler** để chuẩn hóa thang đo.